# Fink LSST — SALT2 Fitting of TNS SNIa Light Curves

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-09
- **last update**: 2026-05-12 : implement error calculation on distance modulus
- **last update**: 2026-05-21 : add SN type in light curves
- **last update**: 2026-05-21 : add flux ratio and magnitude residual panels (section 8b)
- **last update**: 2026-05-21 : add info on dipole at input
## Purpose

This notebook loads the TNS supernova light curves retrieved by
`01_fink_tns_sn_lightcurves.ipynb`, selects objects with a spectroscopic
redshift from TNS (`f:xm_tns_redshift`) and a confirmed or candidate SNIa
classification, and fits their multi-band light curves with the **SALT2**
model using `sncosmo`.

The fitted SALT2 parameters `(t0, x0, x1, c)` are then used to derive the
**distance modulus** μ via the Tripp formula:

$$\mu = m_B - M_B + \alpha\,x_1 - \beta\,c$$

with standard nuisance parameters α = 0.14, β = 3.14, M_B = −19.3 (Betoule+ 2014).

The measured μ(z) is compared to the **ΛCDM prediction** (Ωm = 0.3, ΩΛ = 0.7,
H0 = 70 km/s/Mpc) and to the **empty-universe** reference.

### Data flow
```
01_fink_tns_sn_lightcurves.ipynb
  → data_NB07_01_TNS_SN/catalog_fink_in_tns_lsst.parquet   (object metadata)
  → data_NB07_01_TNS_SN/lc_<diaObjectId>.parquet           (per-object light curves)
        ↓
03_fink_tns_sn_fitSNIa_salt2_lightcurves.ipynb  ← THIS NOTEBOOK
  → SALT2 fit per SNIa with known TNS redshift
  → Hubble diagram μ(z) vs ΛCDM
```

### Key column conventions
- `r:<col>` — original LSST diaSource/diaObject field (prefix = table name, NOT r-band)
- `f:<col>` — Fink-computed field
- Spectral band → `r:band` ∈ {u, g, r, i, z, y}
- Flux unit : **nJy** (nano-Jansky, AB system, ZP = 31.4)

### References
- sncosmo: https://sncosmo.readthedocs.io
- Tripp (1998), Betoule et al. (2014), Rubin DPDD
- Notebook `02_elasticc2_fitsalt2_lightcurves.ipynb` — SALT2 fit pattern on ELAsTiCC2


## 0 — Imports

In [ ]:
import io
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import astropy.table
import requests
import sncosmo
from scipy.integrate import quad

import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Input Data produced by notebook 01_fink_tns_sn_lightcurves.ipynb
NB01_DATA_DIR = Path("data_NB07_01_TNS_SN")
CATALOG_FILE = NB01_DATA_DIR / "catalog_fink_in_tns_lsst.parquet"

# Output directories for this notebook
NB_TAG = "NB07_03_FINK_TNS_SN_FIT_SALT2"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Fink API (used to re-download light curves if needed) ─────────────────────
FINK_API = "https://api.lsst.fink-portal.org/api/v1"
API_DELAY = 0.4  # seconds between requests

# ── Photometric system ────────────────────────────────────────────────────────
# Rubin / Fink LSST alerts: flux in nJy, AB zeropoint ZP = 31.4
# sncosmo convention: flux = F_nu * 10^((ZP - 8.9)/2.5)  with zpsys='ab'
# For nJy: ZP_AB = 2.5*log10(F_0/nJy) + 8.9  where F_0(AB) = 3631 Jy = 3.631e12 nJy
#   → ZP_nJy = 2.5*log10(3.631e12) - 8.9 ≈ 31.4   ✓
RUBIN_ZP = 31.4
ZPSYS = "ab"

# ── SALT2 model ───────────────────────────────────────────────────────────────
SALT2_SOURCE = "salt2-extended"
BAND_PREFIX = "lsst"  # sncosmo LSST band names: lsstu, lsstg, ...
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]

# ── Object selection ──────────────────────────────────────────────────────────
# Only objects with a TNS spectroscopic redshift are fitted
Z_MIN = 0.01  # minimum redshift (avoid nearby peculiar velocity noise)
Z_MAX = 1.5  # maximum redshift
MIN_DATAPOINTS = 5  # minimum number of data points across all bands after quality cuts
SNR_MIN = 3.0  # minimum signal-to-noise ratio per point (psfFlux / psfFluxErr)

# SNIa type strings in TNS (case-insensitive partial match)
# SNIa_TYPES = ["SN Ia", "SN Ia-pec", "SN Ia-91T", "SN Ia-91bg", "SN I", "SN Ic-BL"]
SNIa_TYPES = ["SN Ia", "SN Ia-pec", "SN Ia-91T", "SN Ia-91bg"]
SN_TYPES = ["SN Ia", "SN I", "SN Ib", "SN Ic", "SN II", "SN IIn", "SN Ic-BL", "SLSN-I"]
SN_MARKERS = ["o", "+", "x", "1", "2", "3", "4", "X"]
SN_MARKERS2 = ["o", "s", "D", "^", "v", "<", ">", "X"]

# Set to None to attempt fit on ALL objects with a redshift (including unclassified)
RESTRICT_TO_SNIa = False

# ── Tripp formula nuisance parameters ────────────────────────────────────────
# μ = m_B - M_B + α*x1 - β*c
ALPHA = 0.14  # Betoule+ 2014
BETA = 3.14  # Betoule+ 2014
M_B = -19.3  # absolute magnitude (depends on H0 convention)

# ── ΛCDM cosmology for theoretical Hubble diagram ─────────────────────────────
H0 = 70.0  # km/s/Mpc
OmegaM = 0.30
OmegaL = 0.70
OmegaK = 1.0 - OmegaM - OmegaL  # should be ~0 for flat

# ── Plot colours per band ─────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#3498DB",  # blue
    "g": "#27AE60",  # green
    "r": "#E74C3C",  # red
    "i": "#F1C40F",  # yellow
    "z": "#8B4513",  # brown
    "y": "#95A5A6",  # grey
}

# ── Display grid ──────────────────────────────────────────────────────────────
NCOLS_LC = 3  # columns in the light-curve panel grid

print(f"SALT2 source  : {SALT2_SOURCE}")
print(f"ZP={RUBIN_ZP}  zpsys={ZPSYS}")
print(f"Tripp  α={ALPHA}  β={BETA}  M_B={M_B}")
print(f"ΛCDM   H0={H0}  Ωm={OmegaM}  ΩΛ={OmegaL}")
print(f"Catalog: {CATALOG_FILE}")

## 2 — Load catalog and select SNIa with redshift

The catalog was saved by notebook `01_fink_tns_sn_lightcurves.ipynb`.
We keep only objects with a valid TNS spectroscopic redshift.
Optionally we restrict to TNS-confirmed SNIa types.

In [ ]:
if not CATALOG_FILE.exists():
    raise FileNotFoundError(
        f"Catalog not found: {CATALOG_FILE}\nPlease run notebook 01_fink_tns_sn_lightcurves.ipynb first."
    )

catalog = pd.read_parquet(CATALOG_FILE)
print(f"Catalog loaded: {len(catalog)} objects")
print(f"Columns: {catalog.columns.tolist()}")

In [ ]:
# ── Keep objects with a valid TNS spectroscopic redshift ──────────────────────
tns_z_col = "f:xm_tns_redshift"
tns_type_col = "f:xm_tns_type"
tns_name_col = "f:xm_tns_fullname"
obj_id_col = "r:diaObjectId"
obj_dipole_col = "r:isDipole"

has_z = catalog[tns_z_col].notna() & (catalog[tns_z_col] > 0)
in_range = (catalog[tns_z_col] >= Z_MIN) & (catalog[tns_z_col] <= Z_MAX)
sel = has_z & in_range

if RESTRICT_TO_SNIa:
    is_snia = (
        catalog[tns_type_col].fillna("").apply(lambda t: any(s.lower() in t.lower() for s in SNIa_TYPES))
    )
    sel = sel & is_snia

snia_catalog = catalog[sel].copy().reset_index(drop=True)

print(f"Objects with valid TNS z in [{Z_MIN}, {Z_MAX}]: {has_z.sum()}")
print(f"After {'SNIa type filter' if RESTRICT_TO_SNIa else 'no type filter'}: {len(snia_catalog)}")
snia_catalog[[obj_id_col, tns_name_col, tns_type_col, tns_z_col]].head(20)

## 3 — Download or reload light curves from Fink

For each selected object we fetch the full alert history via `/api/v1/sources`
with `objectId = r:diaObjectId`.  
Light curves are cached locally as parquet files to avoid repeated API calls.

In [ ]:
LC_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
        "r:isDipole",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleMeanFluxErr",
        "r:dipoleNdata",
        "r:dipoleChi2",
        "r:dipoleFitAttempted",
    ]
)


def fetch_lc(obj_id: int, cache_dir: Path, delay: float = API_DELAY) -> pd.DataFrame:
    """Download (or reload from cache) the full alert history for one diaObjectId.

    Parameters
    ----------
    obj_id    : diaObjectId (integer)
    cache_dir : directory to store / look up cached parquet files
    delay     : seconds to wait after each API request (rate limiting)

    Returns
    -------
    DataFrame with light-curve columns, or empty DataFrame on failure
    """
    cache_path = cache_dir / f"lc_{obj_id}.parquet"

    # Return cached version if available
    if cache_path.exists():
        print(f"read {obj_id} from cached file {cache_path}")
        return pd.read_parquet(cache_path)

    # Check if nb01 already cached this object
    nb01_cache = NB01_DATA_DIR / f"lc_{obj_id}.parquet"
    if nb01_cache.exists():
        print(f"read diaobj {obj_id} light curve from cached file {nb01_cache}")
        df = pd.read_parquet(nb01_cache)
        df.to_parquet(cache_path, index=False)
        return df

    # Otherwise download from Fink API
    params = {
        "diaObjectId": obj_id,
        "columns": LC_COLUMNS,
        "output-format": "json",
    }
    try:
        print(f"read {obj_id} from {FINK_API}")
        resp = requests.get(f"{FINK_API}/sources", params=params, timeout=60)
        resp.raise_for_status()
        if not resp.text.strip():
            print(f"  [warn] {obj_id}: empty response")
            return pd.DataFrame()
        df = pd.read_json(io.BytesIO(resp.content))
        df.to_parquet(cache_path, index=False)
        time.sleep(delay)
        return df
    except Exception as exc:
        print(f"  [error] {obj_id}: {exc}")
        return pd.DataFrame()


print(f"fetch_lc helper ready.  Cache → {DATA_DIR}/")

In [ ]:
# Download / reload light curves for all selected objects
lc_dict = {}  # obj_id -> DataFrame

for idx, row in snia_catalog.iterrows():
    oid = int(row[obj_id_col])
    name = row.get(tns_name_col, str(oid))
    stype = row.get(tns_type_col, str(oid))
    df = fetch_lc(oid, DATA_DIR)
    lc_dict[oid] = df
    status = f"{len(df)} pts" if not df.empty else "EMPTY"
    print(f" - {name:20s}({stype}) \t diaObjectId={oid}  → {status}")

print(f"\n{len(lc_dict)} light curves loaded.")

## 4 — Build sncosmo observation tables

Fink LSST alerts deliver flux as **psfFlux** in **nJy** (nano-Jansky), AB system.
The sncosmo zero-point for nJy is ZP = 31.4 (`zpsys='ab'`), consistent with the Rubin
definition `mag = -2.5 * log10(flux_nJy) + 31.4`.

The `r:band` column already contains the single-letter band (u/g/r/i/z/y).  
We prepend `lsst` to match sncosmo's registered LSST bandpass names.

In [ ]:
def make_sncosmo_table_fink(
    lc_df: pd.DataFrame,
    snr_min: float = SNR_MIN,
    zp: float = RUBIN_ZP,
    zpsys: str = ZPSYS,
    band_prefix: str = BAND_PREFIX,
) -> astropy.table.Table:
    """Convert a Fink LSST alert DataFrame to an astropy.Table for sncosmo.

    Input columns expected
    ----------------------
    r:midpointMjdTai  : observation epoch (MJD)
    r:band            : single-letter band name (u/g/r/i/z/y)
    r:psfFlux         : PSF flux in nJy
    r:psfFluxErr      : PSF flux uncertainty in nJy
    r:snr             : signal-to-noise ratio (optional, used for quality cut)
    r:isDipole        : flag is there a dipole

    Returns
    -------
    astropy.Table with columns: time, band, flux, fluxerr, zp, zpsys
    """
    df = lc_df.copy()

    # Rename to generic names
    df = df.rename(
        columns={
            "r:midpointMjdTai": "time",
            "r:band": "band_raw",
            "r:psfFlux": "flux",
            "r:psfFluxErr": "fluxerr",
            "r:snr": "snr",
            "r:isDipole": "isDipole",
        }
    )

    # Quality cuts
    df = df[df["fluxerr"] > 0].copy()
    if "snr" in df.columns:
        df = df[df["snr"] >= snr_min].copy()

    # Build sncosmo band names
    df["band"] = band_prefix + df["band_raw"].str.strip().str.lower()

    table = astropy.table.Table(
        {
            "time": df["time"].values.astype(float),
            "band": df["band"].values,
            "flux": df["flux"].values.astype(float),
            "fluxerr": df["fluxerr"].values.astype(float),
            "zp": np.full(len(df), zp, dtype=float),
            "zpsys": np.full(len(df), zpsys),
            "isDipole": df["isDipole"].values.astype(bool),
        }
    )
    return table


print("make_sncosmo_table_fink helper ready.")

## 5 — SALT2 fit helper

We use `sncosmo.fit_lc` with the `salt2-extended` model.  
The redshift **z is fixed** to the TNS spectroscopic value.  
Free parameters: `t0`, `x0`, `x1`, `c`.

An initial guess for `t0` is taken from the epoch of the peak flux point
among all bands.  An initial guess for `x0` is derived from the peak flux.

In [ ]:
def fit_salt2_fink(
    lc_df: pd.DataFrame,
    z_spec: float,
    fit_z: bool = False,
) -> dict:
    """Fit a Fink LSST SNIa light curve with SALT2 via sncosmo.

    Parameters
    ----------
    lc_df   : raw Fink alert DataFrame for one diaObjectId
    z_spec  : spectroscopic redshift (fixed during fit unless fit_z=True)
    fit_z   : if True, also free the redshift with a tight prior

    Returns
    -------
    dict with keys:
        success, result, fitted_model, table,
        z, t0, x0, x1, c,
        chi2, ndof, chi2_red,
        mB, mu_tripp, message
    """
    # Build the sncosmo observation table
    try:
        obs = make_sncosmo_table_fink(lc_df)
    except Exception as exc:
        return {"success": False, "message": f"Table build failed: {exc}"}

    if len(obs) < MIN_DATAPOINTS:
        return {
            "success": False,
            "message": f"Only {len(obs)} points after quality cuts (need {MIN_DATAPOINTS})",
        }

    # Initialise SALT2 model
    model = sncosmo.Model(source=SALT2_SOURCE)

    # Initial parameter guesses
    peak_idx = int(np.argmax(obs["flux"]))
    t0_guess = float(obs["time"][peak_idx])
    # Rough x0 from peak flux: F_peak ≈ x0 * model_peak  → use 1e-3 as safe default
    model.set(z=z_spec, t0=t0_guess, x0=1e-3, x1=0.0, c=0.0)

    vparam_names = ["t0", "x0", "x1", "c"]
    bounds = {
        "t0": (t0_guess - 40.0, t0_guess + 40.0),
        "x0": (1e-10, 1e2),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }
    if fit_z:
        vparam_names = ["z", "t0", "x0", "x1", "c"]
        dz = min(0.05 * z_spec, 0.1)
        bounds["z"] = (max(z_spec - dz, 1e-4), z_spec + dz)

    # Run fit
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result, fitted_model = sncosmo.fit_lc(
                obs,
                model,
                vparam_names=vparam_names,
                bounds=bounds,
                minsnr=0.0,
                warn=False,
            )
    except Exception as exc:
        return {"success": False, "message": f"Fit failed: {exc}", "table": obs}

    chi2 = result.chisq
    ndof = result.ndof
    chi2_red = chi2 / max(ndof, 1)

    # ── Tripp distance modulus ────────────────────────────────────────────────
    z_fit = float(fitted_model["z"])
    x0_fit = float(fitted_model["x0"])
    x1_fit = float(fitted_model["x1"])
    c_fit = float(fitted_model["c"])
    # m_B = -2.5 * log10(x0) + ZP_SALT2  (SALT2 internal ZP ≈ 10.635)
    # Standard formula: m_B^* = -2.5*log10(x0) + 10.635
    SALT2_ZP_INTERNAL = 10.635
    mB = -2.5 * np.log10(max(x0_fit, 1e-30)) + SALT2_ZP_INTERNAL
    mu = mB - M_B + ALPHA * x1_fit - BETA * c_fit

    return {
        "success": True,
        "result": result,
        "fitted_model": fitted_model,
        "table": obs,
        "z": z_fit,
        "t0": float(fitted_model["t0"]),
        "x0": x0_fit,
        "x1": x1_fit,
        "c": c_fit,
        "chi2": chi2,
        "ndof": ndof,
        "chi2_red": chi2_red,
        "mB": mB,
        "mu_tripp": mu,
        "message": result.message,
    }


print("fit_salt2_fink helper ready.")

In [ ]:
def computemustaterrors(
    param_names, params, covariance, mb_const=10.635, M=-19.3, alpha=0.14, beta=3.1, cosmo=None
):
    """Propagate SALT2 fit covariance into an uncertainty on the Tripp distance modulus.

    Tripp formula
    -------------
        mu  =  m_B*  -  M  +  alpha*x1  -  beta*c
        m_B*  =  -2.5 * log10(x0)  +  mb_const

    Partial derivatives of mu (gradient vector g):
        d(mu)/d(x0) = -2.5 / (x0 * ln10)
        d(mu)/d(x1) =  alpha
        d(mu)/d(c)  = -beta
        d(mu)/d(z)  = (5/ln10)*(1/d_L)*dd_L/dz   [only when z is a free parameter]

    The full error propagation is evaluated as:
        sigma_mu = sqrt( g^T . C_sub . g )
    where C_sub is the covariance sub-matrix restricted to the parameters
    that enter mu (x0, x1, c and optionally z).  t0 is excluded.

    Parameters
    ----------
    param_names : list[str]
        Names of *all* fitted parameters, in the same order as `params` and
        the rows/columns of `covariance`.
        z fixed : ['t0', 'x0', 'x1', 'c']
        z free  : ['z', 't0', 'x0', 'x1', 'c']
    params      : array-like  – fitted values, same order as param_names.
    covariance  : 2-D array   – full covariance matrix, same order.
    mb_const    : SALT2 zero-point offset (default 10.635 for salt2-extended).
    M           : absolute B-band magnitude of a standard SNIa.
    alpha       : Tripp stretch coefficient.
    beta        : Tripp colour coefficient.
    cosmo       : astropy cosmology instance; used only when z is free.
                  Defaults to FlatLambdaCDM(H0=70, Om0=0.3).

    Returns
    -------
    dict with keys:
        mu          – Tripp distance modulus
        sigma_mu    – 1-sigma uncertainty on mu
        gradient    – {param_name: d(mu)/d(param)} for mu-entering params
        sigmas      – {param_name: 1-sigma marginal error} for all params
        covariances – {'pi:pj': cov(pi,pj)} for mu-entering pairs
    """
    params = np.asarray(params, dtype=float)
    cov = np.asarray(covariance, dtype=float)
    names = list(param_names)

    # Build a name→index map so we never hard-code positions
    idx = {name: i for i, name in enumerate(names)}

    # Retrieve the three SALT2 photometric parameters by name
    x0 = params[idx["x0"]]
    x1 = params[idx["x1"]]
    c = params[idx["c"]]

    # Tripp distance modulus
    mB_star = -2.5 * np.log10(x0) + mb_const
    mu = mB_star - M + alpha * x1 - beta * c

    # Gradient of mu w.r.t. the parameters that enter it
    gradient = {
        "x0": -2.5 / (x0 * np.log(10)),  # d(m_B*)/d(x0)
        "x1": alpha,  # d(mu)/d(x1)
        "c": -beta,  # d(mu)/d(c)
    }
    mu_params = ["x0", "x1", "c"]

    # Add the z contribution only when z was a free parameter
    if "z" in idx:
        z = params[idx["z"]]
        if cosmo is None:
            from astropy.cosmology import FlatLambdaCDM

            cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
        dz = 1e-5
        dL = cosmo.luminosity_distance(z).value
        dL_p = cosmo.luminosity_distance(z + dz).value
        dL_m = cosmo.luminosity_distance(z - dz).value
        gradient["z"] = (5.0 / np.log(10)) * (dL_p - dL_m) / (2.0 * dz * dL)
        mu_params.append("z")

    # Gradient vector g and covariance sub-matrix C_sub
    # restricted to the parameters that actually enter mu  (t0 excluded)
    g = np.array([gradient[p] for p in mu_params])
    sub_idx = np.array([idx[p] for p in mu_params])
    C_sub = cov[np.ix_(sub_idx, sub_idx)]

    # sigma_mu^2 = g^T . C_sub . g  (exact first-order error propagation)
    sigma_mu = float(np.sqrt(max(0.0, g @ C_sub @ g)))

    # Marginal 1-sigma errors on every fitted parameter
    sigmas = {name: float(np.sqrt(max(0.0, cov[i, i]))) for name, i in idx.items()}

    # Off-diagonal covariances for mu-entering pairs (upper triangle only)
    covariances = {}
    for ia, a in enumerate(mu_params):
        for ib, b in enumerate(mu_params):
            if ia < ib:
                covariances[f"{a}:{b}"] = float(C_sub[ia, ib])

    return {
        "mu": float(mu),
        "sigma_mu": sigma_mu,
        "gradient": gradient,
        "sigmas": sigmas,
        "covariances": covariances,
    }


print("computemustaterrors ready  (gradient g^T.C.g, param_names-aware)")

## 6 — Check LSST band registration in sncosmo

In [ ]:
salt2_test = sncosmo.Model(source=SALT2_SOURCE)
print(f"SALT2 model loaded: {salt2_test}")
print(f"Parameters: {salt2_test.param_names}")

print("\nChecking LSST band registration in sncosmo:")
for b in BAND_ORDER:
    bname = BAND_PREFIX + b
    try:
        bp = sncosmo.get_bandpass(bname)
        print(f"  {bname:10s}  ✓  {bp}")
    except Exception as exc:
        print(f"  {bname:10s}  ✗  {exc}")

## 7 — Run SALT2 fits

In [ ]:
fit_results = {}  # diaObjectId -> result dict
fit_errors = {}


for idx, row in snia_catalog.iterrows():
    oid = int(row[obj_id_col])
    z_spec = float(row[tns_z_col])
    name = row.get(tns_name_col, str(oid))
    ttype = row.get(tns_type_col, "?")
    lc_df = lc_dict.get(oid, pd.DataFrame())

    if lc_df.empty:
        fit_results[oid] = {"success": False, "message": "No light curve data"}
        print(f"  {name:22s} ({ttype})  diaObj={oid}  ✗  No data")
        continue

    # do the fit here
    res = fit_salt2_fink(lc_df, z_spec=z_spec)
    fit_results[oid] = res

    if res["success"]:
        # retrieve errors
        param_names = res["result"].vparam_names
        params = res["result"].parameters
        cov = res["result"].covariance
        fit_errors[oid] = computemustaterrors(param_names, params, cov)
        sigma_mu = fit_errors[oid]["sigma_mu"]

        print(
            f"  {name:22s}  [{ttype:12s}]  "
            f"z={z_spec:.4f}  t0={res['t0']:.1f}  "
            f"x1={res['x1']:+.3f}  c={res['c']:+.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}  μ={res['mu_tripp']:.2f} "
            f"σ_mu={sigma_mu:.3f}"
        )
    else:
        print(f"  {name:22s}  [{ttype:12s}]  ✗  {res['message']}")
        fit_errors[oid] = None

n_ok = sum(1 for r in fit_results.values() if r["success"])
n_tot = len(fit_results)
print(f"\nSummary: {n_ok}/{n_tot} fits converged.")

## 8 — Plot: multi-band light curves with SALT2 fit

In [ ]:
def plot_salt2_fit_fink(ax, oid: int, name: str, res: dict, z_spec: float):
    """Plot raw Fink data points + SALT2 model curves for one object."""
    if not res.get("success"):
        ax.set_title(f"{name}\nFit failed: {res.get('message', '')}", fontsize=7, color="red")
        ax.text(
            0.5, 0.5, "FIT FAILED", transform=ax.transAxes, ha="center", va="center", color="red", fontsize=12
        )
        return

    obs = res["table"]
    fitted_model = res["fitted_model"]
    t0 = res["t0"]
    chi2_red = res["chi2_red"]

    # Dense time grid for model evaluation
    t_min = float(obs["time"].min())
    t_max = float(obs["time"].max())
    t_dense = np.linspace(t_min - 15, t_max + 15, 500)

    for b in BAND_ORDER:
        bname = BAND_PREFIX + b
        color = BAND_COLORS.get(b, "gray")

        # Data points
        mask = np.array(obs["band"]) == bname
        if mask.sum() > 0:
            ax.errorbar(
                obs["time"][mask] - t0,
                obs["flux"][mask],
                yerr=obs["fluxerr"][mask],
                color=color,
                fmt="o",
                markersize=4,
                capsize=2,
                label=b,
                zorder=3,
            )

        # SALT2 model curve
        try:
            f_model = fitted_model.bandflux(bname, t_dense, zp=RUBIN_ZP, zpsys=ZPSYS)
            valid = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(t_dense[valid] - t0, f_model[valid], color=color, lw=1.5, zorder=2)
        except Exception:
            pass

    ax.axhline(0.0, color="k", lw=0.5, ls="--")
    ax.set_title(
        f"{name}  z={z_spec:.4f}\nx1={res['x1']:+.2f}  c={res['c']:+.2f}  χ²/dof={chi2_red:.2f}",
        fontsize=12,
    )
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=10)
    ax.set_ylabel("psfFlux [nJy]", fontsize=10)
    ax.tick_params(axis="both", labelsize=10)
    ax.legend(fontsize=10, ncol=3, loc="upper right")

In [ ]:
# Build list of successful fits for plotting
ok_list = [
    (oid, snia_catalog.loc[snia_catalog[obj_id_col] == oid].iloc[0])
    for oid in fit_results
    if fit_results[oid]["success"]
]

In [ ]:
n_ok_plots = len(ok_list)
if n_ok_plots == 0:
    print("No successful fits to plot.")
else:
    nrows = math.ceil(n_ok_plots / NCOLS_LC)
    fig, axes = plt.subplots(
        nrows,
        NCOLS_LC,
        figsize=(6.5 * NCOLS_LC, 4.5 * nrows),
        squeeze=False,
    )

    for k, (oid, row) in enumerate(ok_list):
        ax = axes[k // NCOLS_LC][k % NCOLS_LC]
        name = row.get(tns_name_col, str(oid))
        sntype = row.get(tns_type_col, str(oid))
        # add sn type
        name = name + f" ({sntype}) "
        z_s = float(row[tns_z_col])
        plot_salt2_fit_fink(ax, oid, name, fit_results[oid], z_s)

    # Hide empty panels
    for k in range(n_ok_plots, nrows * NCOLS_LC):
        axes[k // NCOLS_LC][k % NCOLS_LC].set_visible(False)

    fig.suptitle(
        "SALT2 fits on Fink/LSST TNS SNIa — psfFlux light curves (nJy)",
        fontsize=12,
        y=1.002,
    )
    fig.tight_layout()
    fig_path = FIGS_DIR / "salt2_fits_lightcurves.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Figure saved → {fig_path}")
    plt.show()

## 8b — Plot: flux ratio data / SALT2 fit  (options a & b)

For each successfully fitted object we show in a sub-panel grid:

* **Option (a)** — Linear flux ratio `F_data / F_model` vs relative time `t − t₀` (centred on 1).
  Error bars are propagated as `σ_ratio = σ_flux / |F_model|`.

* **Option (b)** — Magnitude of the ratio `Δm = −2.5 log₁₀(F_data / F_model)` vs `t − t₀`.
  Error bars are `σ_Δm = (2.5/ln 10) × σ_flux / |F_data|` (standard flux→mag error propagation).

Both options use the same band colour code as in section 8.  
The SN type and reduced χ² of the SALT2 fit are shown in each panel title.

In [ ]:
def plot_flux_ratio_panel(
    ax,
    name: str,
    sntype: str,
    res: dict,
    z_spec: float,
    mode: str = "linear",
) -> None:
    """Plot per-band flux ratio (data / SALT2 fit) for one object.

    Parameters
    ----------
    ax      : matplotlib Axes
    name    : TNS name label
    sntype  : TNS type string (e.g. 'SN Ia')
    res     : result dict from fit_salt2_fink
    z_spec  : spectroscopic redshift
    mode    : 'linear'  → option (a): F_data / F_model  (centred on 1)
              'magnitude' → option (b): Δm = -2.5 log10(F_data/F_model)  (centred on 0)
    """
    if not res.get("success"):
        ax.set_title(f"{name}\nFit failed", fontsize=7, color="red")
        ax.text(
            0.5, 0.5, "FIT FAILED", transform=ax.transAxes, ha="center", va="center", color="red", fontsize=12
        )
        return

    obs = res["table"]
    fitted_model = res["fitted_model"]
    t0 = res["t0"]
    chi2_red = res["chi2_red"]

    for b in BAND_ORDER:
        bname = BAND_PREFIX + b
        color = BAND_COLORS.get(b, "gray")

        mask = np.array(obs["band"]) == bname
        if mask.sum() == 0:
            continue

        t_obs = np.array(obs["time"][mask], dtype=float)
        f_data = np.array(obs["flux"][mask], dtype=float)
        f_err = np.array(obs["fluxerr"][mask], dtype=float)

        # Evaluate SALT2 model at the exact observation epochs
        try:
            f_model = fitted_model.bandflux(bname, t_obs, zp=RUBIN_ZP, zpsys=ZPSYS)
        except Exception:
            continue

        # Keep only points where the model prediction is strictly positive
        valid = np.isfinite(f_model) & (f_model > 0) & (f_data > 0)
        if valid.sum() == 0:
            continue

        dt = t_obs[valid] - t0
        fd = f_data[valid]
        fe = f_err[valid]
        fm = f_model[valid]

        if mode == "linear":
            # Option (a): ratio  R = F_data / F_model
            ratio = fd / fm
            ratio_err = fe / fm  # σ_R = σ_flux / F_model  (model exact)
            ax.errorbar(
                dt,
                ratio,
                yerr=ratio_err,
                color=color,
                fmt="o",
                markersize=4,
                capsize=2,
                label=b,
                zorder=3,
            )
        else:
            # Option (b): magnitude of ratio  Δm = -2.5 log10(F_data / F_model)
            delta_mag = -2.5 * np.log10(fd / fm)
            # Error propagation: σ_Δm = (2.5/ln10) * σ_flux / F_data
            sigma_dm = (2.5 / np.log(10)) * (fe / fd)
            ax.errorbar(
                dt,
                delta_mag,
                yerr=sigma_dm,
                color=color,
                fmt="o",
                markersize=4,
                capsize=2,
                label=b,
                zorder=3,
            )

    # Reference line
    if mode == "linear":
        ax.axhline(1.0, color="k", lw=1.2, ls="--", zorder=2)
        ax.set_ylabel(r"$F_{\rm data}\,/\,F_{\rm model}$", fontsize=10)
    else:
        ax.axhline(0.0, color="k", lw=1.2, ls="--", zorder=2)
        ax.set_ylabel(r"$\Delta m = -2.5\log_{10}(F_{\rm data}/F_{\rm model})$ [mag]", fontsize=9)

    ax.set_title(
        f"{name} ({sntype})  z={z_spec:.3f}\n"
        f"x1={res['x1']:+.2f}  c={res['c']:+.2f}  "
        f"$\\chi^2$/dof={chi2_red:.2f}",
        fontsize=9,
    )
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=10)
    ax.tick_params(axis="both", labelsize=9)
    ax.legend(fontsize=9, ncol=3, loc="upper right")
    ax.grid(True, alpha=0.25)


print("plot_flux_ratio_panel helper ready.")

In [ ]:
# ── Option (a): linear flux ratio F_data / F_model ────────────────────────────
n_ok_plots = len(ok_list)
if n_ok_plots == 0:
    print("No successful fits to plot.")
else:
    nrows = math.ceil(n_ok_plots / NCOLS_LC)
    fig, axes = plt.subplots(
        nrows,
        NCOLS_LC,
        figsize=(6.5 * NCOLS_LC, 4.5 * nrows),
        squeeze=False,
    )

    for k, (oid, row) in enumerate(ok_list):
        ax = axes[k // NCOLS_LC][k % NCOLS_LC]
        name = row.get(tns_name_col, str(oid))
        sntype = row.get(tns_type_col, "?")
        z_s = float(row[tns_z_col])
        plot_flux_ratio_panel(ax, name, sntype, fit_results[oid], z_s, mode="linear")

    # Hide empty panels
    for k in range(n_ok_plots, nrows * NCOLS_LC):
        axes[k // NCOLS_LC][k % NCOLS_LC].set_visible(False)

    fig.suptitle(
        r"SALT2 fit residuals (a) — linear flux ratio $F_{\rm data}\,/\,F_{\rm model}$"
        " — Fink/LSST TNS SN",
        fontsize=12,
        y=1.002,
    )
    fig.tight_layout()
    fig_path_a = FIGS_DIR / "salt2_flux_ratio_linear.png"
    fig.savefig(fig_path_a, dpi=150, bbox_inches="tight")
    print(f"Figure (a) saved → {fig_path_a}")
    plt.show()

In [ ]:
# ── Option (b): magnitude of flux ratio  Δm = -2.5 log10(F_data / F_model) ───
n_ok_plots = len(ok_list)
if n_ok_plots == 0:
    print("No successful fits to plot.")
else:
    nrows = math.ceil(n_ok_plots / NCOLS_LC)
    fig, axes = plt.subplots(
        nrows,
        NCOLS_LC,
        figsize=(6.5 * NCOLS_LC, 4.5 * nrows),
        squeeze=False,
    )

    for k, (oid, row) in enumerate(ok_list):
        ax = axes[k // NCOLS_LC][k % NCOLS_LC]
        name = row.get(tns_name_col, str(oid))
        sntype = row.get(tns_type_col, "?")
        z_s = float(row[tns_z_col])
        plot_flux_ratio_panel(ax, name, sntype, fit_results[oid], z_s, mode="magnitude")

    # Hide empty panels
    for k in range(n_ok_plots, nrows * NCOLS_LC):
        axes[k // NCOLS_LC][k % NCOLS_LC].set_visible(False)

    fig.suptitle(
        r"SALT2 fit residuals (b) — $\Delta m = -2.5\log_{10}(F_{\rm data}/F_{\rm model})$"
        " — Fink/LSST TNS SN",
        fontsize=12,
        y=1.002,
    )
    fig.tight_layout()
    fig_path_b = FIGS_DIR / "salt2_flux_ratio_magnitude.png"
    fig.savefig(fig_path_b, dpi=150, bbox_inches="tight")
    print(f"Figure (b) saved → {fig_path_b}")
    plt.show()

## 9 — SALT2 parameter distributions

In [ ]:
# Build a summary DataFrame from successful fits
rows = []
for oid, res in fit_results.items():
    if not res["success"]:
        continue
    meta = snia_catalog.loc[snia_catalog[obj_id_col] == oid].iloc[0]

    sigma_mu = fit_errors[oid]["sigma_mu"]

    rows.append(
        {
            "diaObjectId": oid,
            "tns_name": meta.get(tns_name_col, ""),
            "tns_type": meta.get(tns_type_col, ""),
            "z_spec": float(meta[tns_z_col]),
            "z_fit": res["z"],
            "t0": res["t0"],
            "x0": res["x0"],
            "x1": res["x1"],
            "c": res["c"],
            "mB": res["mB"],
            "mu_tripp": res["mu_tripp"],
            "chi2_red": res["chi2_red"],
            "ndof": res["ndof"],
            "sigma_mu": sigma_mu,
        }
    )

results_df = pd.DataFrame(rows)
print(f"{len(results_df)} successful fits.")
results_df.to_parquet(DATA_DIR / "salt2_fit_results.parquet", index=False)
results_df

In [ ]:
if len(results_df) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    # x1
    axes[0].hist(results_df["x1"], bins=15, color="steelblue", edgecolor="white")
    axes[0].axvline(0.0, color="k", ls="--", lw=1)
    axes[0].set_xlabel(r"$x_1$ (stretch)", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title(r"SALT2 stretch $x_1$")

    # c
    axes[1].hist(results_df["c"], bins=15, color="tomato", edgecolor="white")
    axes[1].axvline(0.0, color="k", ls="--", lw=1)
    axes[1].set_xlabel(r"$c$ (colour)", fontsize=12)
    axes[1].set_ylabel("Count", fontsize=12)
    axes[1].set_title(r"SALT2 colour $c$")

    # χ²/dof
    axes[2].hist(results_df["chi2_red"], bins=15, color="seagreen", edgecolor="white")
    axes[2].axvline(1.0, color="k", ls="--", lw=1, label="χ²/dof = 1")
    axes[2].set_xlabel(r"$\chi^2 / \mathrm{dof}$", fontsize=12)
    axes[2].set_ylabel("Count", fontsize=12)
    axes[2].set_title(r"Reduced $\chi^2$")
    axes[2].legend()

    fig.suptitle("SALT2 parameter distributions — Fink/LSST TNS SNIa", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "salt2_parameter_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough successful fits to plot distributions.")

## 10 — Theoretical ΛCDM distance modulus

The luminosity distance in a flat ΛCDM cosmology is:

$$d_L(z) = \frac{c(1+z)}{H_0} \int_0^z \frac{dz'}{E(z')}$$

where $E(z) = \sqrt{\Omega_m (1+z)^3 + \Omega_\Lambda}$ (flat, no radiation).  
The distance modulus is then:

$$\mu_{\rm th}(z) = 5 \log_{10}\!\left(\frac{d_L}{10\,\mathrm{pc}}\right)
                  = 5 \log_{10}\!\left(\frac{d_L}{1\,\mathrm{Mpc}}\right) + 25$$

We also show the **empty-universe** reference ($\Omega_m = 0,\,\Omega_\Lambda = 0$),
for which $d_L = c z (1+z/2)/H_0$ (linear approximation for small z, exact integral here).

In [ ]:
C_LIGHT = 2.998e5  # km/s


def E_func(z, OmegaM=OmegaM, OmegaL=OmegaL):
    """Dimensionless Hubble parameter E(z) = H(z)/H0 for flat ΛCDM."""
    return np.sqrt(OmegaM * (1.0 + z) ** 3 + OmegaL)


def comoving_distance_Mpc(z, H0=H0, OmegaM=OmegaM, OmegaL=OmegaL):
    """Comoving distance in Mpc for a flat ΛCDM cosmology."""
    integrand = lambda zp: 1.0 / E_func(zp, OmegaM, OmegaL)
    chi, _ = quad(integrand, 0.0, z)
    return (C_LIGHT / H0) * chi


def distance_modulus_lcdm(z_arr, H0=H0, OmegaM=OmegaM, OmegaL=OmegaL):
    """ΛCDM theoretical distance modulus μ(z) for an array of redshifts."""
    mu_arr = np.zeros_like(z_arr, dtype=float)
    for i, z in enumerate(z_arr):
        if z <= 0:
            mu_arr[i] = np.nan
            continue
        dc = comoving_distance_Mpc(z, H0, OmegaM, OmegaL)
        dl = dc * (1.0 + z)  # luminosity distance [Mpc]
        mu_arr[i] = 5.0 * np.log10(dl) + 25.0  # convert Mpc → pc: +25
    return mu_arr


def distance_modulus_empty(z_arr, H0=H0):
    """Empty-universe distance modulus (Ωm=0, ΩΛ=0)."""
    return distance_modulus_lcdm(z_arr, H0=H0, OmegaM=0.0, OmegaL=0.0)


# Quick sanity check at z=0.5
z_test = 0.5
mu_test = distance_modulus_lcdm(np.array([z_test]))[0]
print(f"ΛCDM μ(z={z_test}) = {mu_test:.3f}  (expected ≈ 42.4 for H0=70, Ωm=0.3)")

In [ ]:
results_df

## 11 — Hubble diagram: μ(z) measured vs ΛCDM

In [ ]:
if len(results_df) == 0:
    print("No successful fits — Hubble diagram cannot be drawn.")
else:
    # ── Quality cut for Hubble diagram ─────────────────────────────────────────
    # Keep only well-converged fits: χ²/dof < 3 and |x1| < 4 and |c| < 0.4
    # hd = results_df[
    #    # (results_df["chi2_red"] < 3.0) & (results_df["x1"].abs() < 4.0) & (results_df["c"].abs() < 0.4)
    #    (results_df["chi2_red"] < 10.0) & (results_df["x1"].abs() < 6.0) & (results_df["c"].abs() < 1.0)
    # ].copy()

    hd = results_df.copy()
    print(f"Hubble diagram: {len(hd)}/{len(results_df)} events after quality cuts.")

    # ── Theoretical curves ─────────────────────────────────────────────────────
    z_theory = np.linspace(1e-3, Z_MAX * 1.05, 300)
    mu_lcdm = distance_modulus_lcdm(z_theory)
    mu_empty = distance_modulus_empty(z_theory)

    # ── Hubble diagram plot ────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(9, 7),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True,
    )

    ax_hub = axes[0]
    ax_res = axes[1]

    # --- upper panel: μ(z) ---
    ax_hub.plot(
        z_theory,
        mu_lcdm,
        color="royalblue",
        lw=2.0,
        ls="-",
        label=rf"ΛCDM ($\Omega_m={OmegaM}$, $\Omega_\Lambda={OmegaL}$, $H_0={H0}$)",
    )

    ax_hub.errorbar(
        hd["z_spec"],
        hd["mu_tripp"],
        yerr=hd["sigma_mu"],
        lw=0,
        marker=".",
        color="grey",
        ecolor="grey",
        capsize=2,
        elinewidth=1,
        alpha=1,
    )

    for sn_type, marker in zip(SN_TYPES, SN_MARKERS):
        subset = hd[hd["tns_type"] == sn_type]
        if len(subset) > 0:
            if sn_type == "SN Ia":
                sc = ax_hub.scatter(
                    subset["z_spec"],
                    subset["mu_tripp"],
                    c=subset["chi2_red"],
                    cmap="jet",
                    vmin=0,
                    vmax=5,
                    s=80,
                    linewidths=0.5,
                    marker=marker,
                    label=sn_type,
                )
            else:
                sc = ax_hub.scatter(
                    subset["z_spec"],
                    subset["mu_tripp"],
                    c=subset["chi2_red"],
                    cmap="jet",
                    vmin=0,
                    vmax=5,
                    s=40,
                    facecolors="none",
                    linewidths=0.5,
                    marker=marker,
                    label=sn_type,
                )

    plt.colorbar(sc, ax=ax_hub, label=r"$\chi^2$/dof")

    ax_hub.legend(loc="upper right")

    ax_hub.set_ylabel(r"Distance modulus $\mu$", fontsize=12)
    ax_hub.legend(fontsize=9)
    ax_hub.set_title(
        "Hubble diagram — Fink/LSST TNS SN (SALT2 fit, Tripp formula)",
        fontsize=12,
    )
    ax_hub.grid(True, alpha=0.3)

    ax_hub.set_xlim(0, Z_MAX)

    # --- lower panel: residuals Δμ = μ_measured − μ_ΛCDM ---

    ax_res.axhline(0.0, color="royalblue", lw=1.5, ls="-")

    if len(hd) > 0:
        # Interpolate ΛCDM at the observed redshifts
        mu_lcdm_obs = distance_modulus_lcdm(hd["z_spec"].values)

        ax_res.errorbar(
            hd["z_spec"],
            hd["mu_tripp"] - mu_lcdm_obs,
            yerr=hd["sigma_mu"],
            lw=0,
            marker=".",
            color="grey",
            ecolor="grey",
            capsize=2,
            elinewidth=1,
            alpha=1,
        )

        for sn_type, marker in zip(SN_TYPES, SN_MARKERS):
            subset = hd[hd["tns_type"] == sn_type]
            if len(subset) > 0:
                mu_lcdm_obs = distance_modulus_lcdm(subset["z_spec"].values)
                residuals = subset["mu_tripp"].values - mu_lcdm_obs

                if sn_type == "SN Ia":
                    sc = ax_res.scatter(
                        subset["z_spec"],
                        residuals,
                        c=subset["chi2_red"],
                        cmap="jet",
                        vmin=0,
                        vmax=5,
                        s=80,
                        linewidths=0.5,
                        marker=marker,
                        label=sn_type,
                    )
                else:
                    sc = ax_res.scatter(
                        subset["z_spec"],
                        residuals,
                        c=subset["chi2_red"],
                        cmap="jet",
                        vmin=0,
                        vmax=5,
                        s=40,
                        facecolors="none",
                        linewidths=0.5,
                        marker=marker,
                        label=sn_type,
                    )

    plt.colorbar(sc, ax=ax_res, label=r"$\chi^2$/dof")

    # ax_res.axhline(0.0, color="k", lw=0.8, ls="--")

    ax_res.set_xlabel("Redshift $z$", fontsize=12)
    ax_res.set_ylabel(r"$\Delta\mu$ [mag]", fontsize=12)
    ax_res.set_ylim(-3.5, 3.5)
    ax_res.grid(True, alpha=0.3)
    ax_res.set_xlim(0, Z_MAX)

    fig.tight_layout()
    fig_path = FIGS_DIR / "hubble_diagram_lcdm.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Hubble diagram saved → {fig_path}")
    plt.show()

## 12 — Colour–stretch (x1 vs c) plane

In [ ]:
def get_errors(subset, fit_results, fit_errors, x_name="c", y_name="x1"):
    """
    Extracts the values and errors of the fitted parameters x='c' and y='x1' for the given subset of data.

    Parameters:
    subset (DataFrame): A subset of the DataFrame containing the data for which to extract errors.
    fit_errors (dict): A dictionary containing the fit errors for each object.

    Returns:
    X (array): Array of 'c' values.
    Y (array): Array of 'x1' values.
    ErrX (array): Array of errors for 'c' values.
    ErrY (array): Array of errors for 'x1' values.
    """
    oids = subset["diaObjectId"].values

    X = []
    Y = []
    ErrX = []
    ErrY = []

    for oid in oids:
        if oid in fit_errors and fit_errors[oid] is not None:
            # Extract 'c' and 'x1' values
            x_value = fit_results[oid][x_name]
            y_value = fit_results[oid][y_name]

            # Extract errors for 'c' and 'x1'
            x_error = fit_errors[oid]["sigmas"][x_name]
            y_error = fit_errors[oid]["sigmas"][y_name]

            X.append(x_value)
            Y.append(y_value)
            ErrX.append(x_error)
            ErrY.append(y_error)

    return np.array(X), np.array(Y), np.array(ErrX), np.array(ErrY)

In [ ]:
if len(results_df) >= 2:
    fig, ax = plt.subplots(figsize=(7, 5))

    for sn_type, marker in zip(SN_TYPES, SN_MARKERS2):
        subset = results_df[results_df["tns_type"] == sn_type]
        if len(subset) > 0:
            # get errors
            x, y, xerr, yerr = get_errors(subset, fit_results, fit_errors)

            if sn_type == "SN Ia":
                ax.errorbar(
                    x,
                    y,
                    xerr=xerr,
                    yerr=yerr,
                    lw=0,
                    color="k",
                    marker=".",
                    ecolor="grey",
                    capsize=2,
                    elinewidth=1,
                    alpha=1,
                )

                sc = ax.scatter(
                    subset["c"],
                    subset["x1"],
                    c=subset["z_spec"],
                    cmap="jet",
                    vmin=0,
                    vmax=Z_MAX,
                    s=80,
                    marker=marker,
                    lw=0.4,
                    zorder=1,
                    label=sn_type,
                )

            else:
                ax.errorbar(
                    x,
                    y,
                    xerr=xerr,
                    yerr=yerr,
                    lw=0,
                    color="grey",
                    marker=".",
                    ecolor="grey",
                    capsize=2,
                    elinewidth=1,
                    alpha=1,
                )
                sc = ax.scatter(
                    subset["c"],
                    subset["x1"],
                    c=subset["z_spec"],
                    cmap="jet",
                    vmin=0,
                    vmax=Z_MAX,
                    s=20,
                    marker=marker,
                    facecolors="none",
                    lw=0.4,
                    zorder=4,
                    label=sn_type,
                )

    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Spectroscopic redshift $z$", fontsize=11)
    ax.legend(loc="best")

    ax.axvline(0.0, color="grey", ls="--", lw=0.8)
    ax.axhline(0.0, color="grey", ls="--", lw=0.8)
    ax.set_xlabel(r"SALT2 colour $c$", fontsize=12)
    ax.set_ylabel(r"SALT2 stretch $x_1$", fontsize=12)
    ax.set_title("Colour–stretch plane — Fink/LSST TNS SN", fontsize=12)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "salt2_colour_stretch_plane.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough data for colour–stretch plot.")

## 13 — Summary table

In [ ]:
if len(results_df) > 0:
    display_cols = [
        "tns_name",
        "tns_type",
        "z_spec",
        "x1",
        "c",
        "mB",
        "mu_tripp",
        "chi2_red",
        "ndof",
    ]
    print(results_df[display_cols].to_string(index=False, float_format="{:.4f}".format))
else:
    print("No successful fits.")

## 14 — Save results

In [ ]:
if len(results_df) > 0:
    out_parquet = DATA_DIR / "salt2_fit_results.parquet"
    out_csv = DATA_DIR / "salt2_fit_results.csv"
    results_df.to_parquet(out_parquet, index=False)
    results_df.to_csv(out_csv, index=False)
    print(f"Results saved:\n  {out_parquet}\n  {out_csv}")
else:
    print("Nothing to save.")

---
## Notes and caveats

- **Redshift**: fixed to `f:xm_tns_redshift` (spectroscopic). If this is NaN or
  wrong, the fit will fail or produce unphysical parameters.
- **ZP convention**: Fink LSST psfFlux is in nJy. The SALT2 fit uses
  ZP = 31.4, zpsys = 'ab', consistent with the Rubin AB definition.
- **Tripp calibration**: α, β, M_B are fixed to Betoule+ 2014 values.
  A proper cosmological analysis requires simultaneous fitting of the
  nuisance parameters and the cosmological model.
- **χ²/dof > 1**: expected for real data with correlated noise, PSF
  modelling residuals, and host-galaxy contamination.
- **Light-curve sampling**: Rubin in its first year may not yet provide
  the full pre-/post-maximum coverage needed for a well-constrained SALT2 fit.
  The fit may fail or be poorly constrained for objects observed only near peak.


## Save supernovae TNS

In [ ]:
df_snIa = results_df[results_df["tns_type"] == "SN Ia"]

In [ ]:
df_snIa

In [ ]:
df_snIa.to_csv("snIa_TNS.csv")